# E4c — ce que coûte le choix du seuil> **Notebook v1, compilé le 2026-08-15.** La cellule 1 réaffiche ce numéro.**Prérequis : E4a-bis doit avoir tourné et confirmé la liste noire.****La critique à laquelle ce notebook répond.** Le manuscrit montre déjà que la liste noirepasse de huit à treize colonnes comportementales selon la normalisation, et qu'elle bougeavec le seuil. Mais il n'a jamais réentraîné sous ces alternatives : il publie des listesde features, pas des conséquences. Tant que c'est le cas, la règle reste un diagnosticproposé, et le relecteur a raison de refuser de la traiter comme une méthode validée.Ce notebook réentraîne sous **quatre listes** et mesure ce que chacune coûte.| Variante | Règle | Comportementales exclues ||---|---|---|| `clean` | aucune exclusion comportementale | 0 || `tau050` | τ < 0.50 — **la règle publiée** | 8 || `tau040` | τ < 0.40 | 6 || `tau070` | τ < 0.70 | 16 || `taustar` | τ\* < 0.50, corrigé du hasard | 13 |Pour chaque variante, chaque détecteur est réentraîné sous les **deux protocoles**, et onrapporte macro-F1 stratifié, macro-F1 temporel, FPR et ECE.**La question à laquelle le tableau final répond :** si les cinq variantes donnent les mêmesscores, le choix du seuil est sans conséquence et la règle est robuste. Si elles divergent,le manuscrit doit publier l'intervalle plutôt qu'un chiffre.**Durée.** 4 variantes nouvelles × 10 détecteurs × 2 protocoles = 80 runs, dont 24réseaux. Compter 8 à 12 h avec GPU. Le notebook est reprenable : on peut le relancerautant de fois qu'il faut, il saute ce qui est déjà fait.**À me renvoyer :** `e4c_results.json`.

In [ ]:
# --- 1. Drive, dossier, verrou E4a ---------------------------------------E4C_VERSION = "v1"; E4C_BUILD = "2026-08-15"print(f"E4c notebook {E4C_VERSION}, compile le {E4C_BUILD}\n")import pathlib, json, sysfrom google.colab import drivedrive.mount("/content/drive")MYDRIVE = pathlib.Path("/content/drive/MyDrive")MARQUEUR = "article1_results.json"candidats = [MYDRIVE / "GeNIS" / "article1_final", MYDRIVE / "article1_final",             MYDRIVE / "GeNIS"]trouves = [c for c in candidats if (c / MARQUEUR).exists()]if not trouves:    for prof in ("*/", "*/*/", "*/*/*/"):        trouves = [p.parent for p in MYDRIVE.glob(prof + MARQUEUR)]        if trouves:            breakif not trouves:    sys.exit(f"{MARQUEUR} introuvable sous {MYDRIVE}.")SAVE = trouves[0]print(f"dossier de travail : {SAVE}")# E4a a montre que tau ne peut pas se calculer sur la validation, qui est# adjacente au train et sous-estime la decroissance. C'est donc E4a-bis,# l'audit imbrique dans la partition d'entrainement, qui fait foi. Le verrou# recalcule le classement depuis les donnees plutot que de lire un booleen.E4AB = SAVE / "e4abis_results.json"if not E4AB.exists():    sys.exit("e4abis_results.json introuvable : lance E4a-bis d'abord. Le "             "verdict de E4a seul ne suffit pas ; voir experiments/e4a/FINDINGS.md")_B = json.loads(E4AB.read_text(encoding="utf-8"))_R = json.loads((SAVE / MARQUEUR).read_text(encoding="utf-8"))_N = _B["nested"]_PUB = sorted(f for f in _R["audit"]["blacklist"] if f in _N)_P, _K = _R["audit"]["chance"], _R["audit"]["rule"]["min_acc_x_chance"]_elig = [f for f in _N if _N[f]["strat"] > _K * _P]_srt = sorted(_elig, key=lambda f: _N[f]["tau"])_taus = [_N[f]["tau"] for f in _srt]_gmax, _imax = max((_taus[i + 1] - _taus[i], i) for i in range(len(_taus) - 1))_ok_liste = set(_srt[:len(_PUB)]) == set(_PUB)_ok_ecart = _imax + 1 == len(_PUB)print(f"audit imbrique : {len(_elig)} eligibles, {len(_PUB)} colonnes publiees")print(f"   les {len(_PUB)} publiees sont-elles les {len(_PUB)} plus basses ? "      f"{'OUI' if _ok_liste else 'NON'}")print(f"   le plus large ecart ({_gmax:.4f}) tombe-t-il apres le rang {len(_PUB)} ? "      f"{'OUI' if _ok_ecart else 'NON, apres le rang %d' % (_imax + 1)}")if not _ok_liste:    print(f"   classement imbrique : {_srt[:len(_PUB) + 2]}")    sys.exit("\nARRET. L'audit imbrique ne retrouve pas la liste publiee : le "             "benchmark doit d'abord etre relance sur la liste qu'il retourne.")if not _ok_ecart:    print("   avertissement : la liste est retrouvee mais la coupure naturelle "          "ne tombe pas exactement apres elle. On continue.")print("verrou leve : la liste noire est confirmee sans information de test.\n")

In [ ]:
# --- 2. Donnees, et les cinq listes noires -------------------------------import numpy as np, pandas as pd, time, gcfrom sklearn.preprocessing import RobustScalerfrom sklearn.metrics import f1_score, matthews_corrcoefR = json.loads((SAVE / "article1_results.json").read_text(encoding="utf-8"))M = json.loads((SAVE / "cache" / "slice60_meta.json").read_text(encoding="utf-8"))z = np.load(SAVE / "cache" / "slice60.npz")X, y = z["X"], z["y"].astype(int)FEATS = M["feat_all"]F_CLEAN = R["slice60"]["features_clean"]CLASS_NAMES = R["slice60"]["classes"]; C = len(CLASS_NAMES)BENIGN = CLASS_NAMES.index("benign")P_CHANCE = R["audit"]["chance"]KMIN = R["audit"]["rule"]["min_acc_x_chance"]TT = {r["feature"]: r for r in R["audit"]["transfer_table"]}zs = np.load(SAVE / "frozen_splits_60s.npz")SPLITS = {k: (zs[f"{k}_train"], zs[f"{k}_val"], zs[f"{k}_test"])          for k in ("strat_seed1", "temporal")}def exclues(seuil, corrige):    out = []    for f in F_CLEAN:        r = TT[f]; s_, t_ = r["acc seule (stratifie)"], r["acc seule (temporel)"]        if s_ <= KMIN * P_CHANCE:            continue        ratio = ((t_ - P_CHANCE) / (s_ - P_CHANCE)) if corrige else (t_ / s_)        if ratio < seuil:            out.append(f)    return sorted(out)VARIANTES = {"clean":   [],             "tau040":  exclues(0.40, False),             "tau050":  exclues(0.50, False),             "tau070":  exclues(0.70, False),             "taustar": exclues(0.50, True)}print(f"{'variante':<10}{'exclues':>9}   features conservees")for k, v in VARIANTES.items():    print(f"{k:<10}{len(v):>9}   {len(F_CLEAN)-len(v)}")pub = sorted(f for f in R["audit"]["blacklist"] if f in F_CLEAN)print(f"\ncontrole : tau050 reproduit la liste publiee : "      f"{'OUI' if VARIANTES['tau050'] == pub else 'NON'}")if VARIANTES["tau050"] != pub:    print(f"   calculee : {VARIANTES['tau050']}")    print(f"   publiee  : {pub}")STATE_PATH = SAVE / "e4c_results.json"def load_state():    if STATE_PATH.exists():        return json.loads(STATE_PATH.read_text(encoding="utf-8"))    return {"meta": {"version": E4C_VERSION},            "variantes": {k: v for k, v in VARIANTES.items()}, "runs": {}}def save_state(s):    tmp = STATE_PATH.with_suffix(".tmp")    tmp.write_text(json.dumps(s, indent=1, ensure_ascii=False, default=float),                   encoding="utf-8")    tmp.replace(STATE_PATH)STATE = load_state()print(f"\netat : {len(STATE['runs'])} runs deja faits")

In [ ]:
# --- 3. Detecteurs, ECE, evaluation --------------------------------------from sklearn.dummy import DummyClassifierfrom sklearn.linear_model import LogisticRegressionfrom sklearn.naive_bayes import GaussianNBfrom sklearn.neighbors import KNeighborsClassifierfrom sklearn.ensemble import RandomForestClassifierfrom xgboost import XGBClassifierfrom lightgbm import LGBMClassifierfrom sklearn.utils.class_weight import compute_class_weightimport tensorflow as tffrom tensorflow import kerasfrom tensorflow.keras import layers, models, callbacksKNN_MAX_TRAIN = 50_000SK = ["majority", "logreg", "nb", "knn", "rf", "xgboost", "lightgbm"]DEEP = ["dnn", "cnn", "rnn"]MODELES = SK + DEEPdef make_sk(name):    if name == "majority": return DummyClassifier(strategy="most_frequent")    if name == "logreg":   return LogisticRegression(max_iter=1000, n_jobs=-1)    if name == "nb":       return GaussianNB()    if name == "knn":      return KNeighborsClassifier(n_neighbors=5, n_jobs=2)    if name == "rf":       return RandomForestClassifier(n_estimators=200, n_jobs=2,                                                         random_state=0)    if name == "xgboost":  return XGBClassifier(n_estimators=300, max_depth=8,                                                learning_rate=.1, tree_method="hist",                                                n_jobs=2, random_state=0,                                                eval_metric="mlogloss")    if name == "lightgbm": return LGBMClassifier(n_estimators=300, num_leaves=63,                                                 learning_rate=.1, n_jobs=2,                                                 random_state=0, verbose=-1)    raise KeyError(name)def build_deep(name, F):    if name == "dnn":        return models.Sequential([layers.Input((F,)),            layers.Dense(128, activation="relu"), layers.Dropout(.3),            layers.Dense(64, activation="relu"), layers.Dropout(.2),            layers.Dense(C, activation="softmax")], name="dnn")    if name == "cnn":        return models.Sequential([layers.Input((F, 1)),            layers.Conv1D(64, 3, activation="relu", padding="same"),            layers.Conv1D(32, 3, activation="relu", padding="same"),            layers.GlobalMaxPooling1D(), layers.Dropout(.3),            layers.Dense(64, activation="relu"),            layers.Dense(C, activation="softmax")], name="cnn")    if name == "rnn":        return models.Sequential([layers.Input((F, 1)),            layers.SimpleRNN(64), layers.Dropout(.3),            layers.Dense(64, activation="relu"),            layers.Dense(C, activation="softmax")], name="rnn")    raise KeyError(name)def ece(probs, yt, bins=15):    conf = probs.max(1); pred = probs.argmax(1); ok = (pred == yt).astype(float)    edges = np.linspace(0, 1, bins + 1); e = 0.0    for i in range(bins):        m = (conf > edges[i]) & (conf <= edges[i + 1])        if m.any():            e += m.mean() * abs(ok[m].mean() - conf[m].mean())    return float(e)def brier(probs, yt):    oh = np.zeros_like(probs); oh[np.arange(len(yt)), yt] = 1.0    return float(((probs - oh) ** 2).sum(1).mean())def evalue(yt, pr):    p = pr.argmax(1); att, patt = yt != BENIGN, p != BENIGN    return {"macro_f1": float(f1_score(yt, p, average="macro", zero_division=0)),            "accuracy": float((p == yt).mean()),            "mcc": float(matthews_corrcoef(yt, p)),            "fpr": float(patt[~att].mean()) if (~att).any() else None,            "ece": ece(pr, yt), "brier": brier(pr, yt)}def fit_predict(name, Xtr, ytr, Xva, yva, Xte):    if name in SK:        if name == "knn" and len(ytr) > KNN_MAX_TRAIN:            rng = np.random.RandomState(0)            keep = rng.choice(len(ytr), KNN_MAX_TRAIN, replace=False)            Xtr, ytr = Xtr[keep], ytr[keep]        clf = make_sk(name).fit(Xtr, ytr)        pr = clf.predict_proba(Xte)        if pr.shape[1] != C:            full = np.zeros((len(Xte), C));            for j, c_ in enumerate(clf.classes_):                full[:, int(c_)] = pr[:, j]            pr = full        del clf; gc.collect(); return pr    tf.keras.utils.set_random_seed(1)    net = build_deep(name, Xtr.shape[1])    net.compile(optimizer=keras.optimizers.Adam(1e-3),                loss="sparse_categorical_crossentropy")    cls = np.unique(ytr); w = compute_class_weight("balanced", classes=cls, y=ytr)    net.fit(Xtr, ytr, validation_data=(Xva, yva), epochs=30, batch_size=256, verbose=0,            class_weight={int(c_): float(v) for c_, v in zip(cls, w)},            callbacks=[callbacks.EarlyStopping(monitor="val_loss", patience=5,                                               restore_best_weights=True)])    pr = net.predict(Xte, batch_size=2048, verbose=0)    del net; keras.backend.clear_session(); gc.collect(); return prprint("detecteurs et metriques definis")

In [ ]:
# --- 4. La campagne ------------------------------------------------------A_FAIRE = ["tau040", "tau070", "taustar", "clean"]   # tau050 = deja publiet_global = time.time()for var in A_FAIRE:    cols = [c for c in F_CLEAN if c not in VARIANTES[var]]    ci = [FEATS.index(c) for c in cols]    print(f"\n{'='*70}\nvariante {var} : {len(cols)} features\n{'='*70}", flush=True)    for proto in ("strat_seed1", "temporal"):        tr_, va_, te_ = SPLITS[proto]        Xtr = Xva = Xte = None        for mod in MODELES:            cle = f"{mod}|{var}|{proto}"            if cle in STATE["runs"]:                continue            if Xtr is None:                sc = RobustScaler().fit(X[tr_][:, ci])                g = lambda ix: np.nan_to_num(sc.transform(X[ix][:, ci]),                                             nan=0., posinf=0., neginf=0.).astype("float32")                Xtr, Xva, Xte = g(tr_), g(va_), g(te_)            t1 = time.time()            pr = fit_predict(mod, Xtr, y[tr_], Xva, y[va_], Xte)            r = evalue(y[te_], pr); r["seconds"] = round(time.time() - t1, 1)            r["n_features"] = len(cols)            STATE["runs"][cle] = r; save_state(STATE)            print(f"   [{proto:11s}] {mod:10s} mF1 {r['macro_f1']:.4f}  "                  f"ECE {r['ece']:.5f}  ({r['seconds']:.0f} s)", flush=True)            del pr; gc.collect()        del Xtr, Xva, Xte; gc.collect()print(f"\ncampagne terminee en {(time.time()-t_global)/3600:.1f} h")

In [ ]:
# --- 5. Le tableau qui repond a la critique 3 ----------------------------def val(mod, var, proto, champ="macro_f1"):    if var == "tau050":            # la regle publiee : on lit la campagne d'origine        k = f"{mod}|audited|{proto}"        return R["models"].get(k, {}).get(champ)    return STATE["runs"].get(f"{mod}|{var}|{proto}", {}).get(champ)ORDRE = ["clean", "tau040", "tau050", "tau070", "taustar"]for champ, titre in (("macro_f1", "macro-F1"), ("ece", "ECE"), ("fpr", "FPR")):    for proto in ("strat_seed1", "temporal"):        print(f"\n=== {titre} — {proto} ===")        print(f"{'detecteur':<12}" + "".join(f"{v:>11}" for v in ORDRE))        print("-" * (12 + 11 * len(ORDRE)))        for mod in MODELES:            row = ""            for v in ORDRE:                x = val(mod, v, proto, champ)                row += f"{x:>11.4f}" if isinstance(x, (int, float)) else f"{'--':>11}"            print(f"{mod:<12}{row}")print("\n" + "=" * 70)print("LECTURE : pour chaque detecteur, l'ecart entre la colonne tau050 (regle")print("publiee) et les autres est le cout du choix de seuil. S'il reste sous")print("l'ecart-type inter-graines du manuscrit, le choix est sans consequence")print("et la regle est robuste. Sinon, le manuscrit doit publier l'intervalle.")print("=" * 70)ecarts = []for mod in MODELES:    if mod == "majority":        continue    ref = val(mod, "tau050", "temporal")    for v in ORDRE:        x = val(mod, v, "temporal")        if isinstance(x, (int, float)) and isinstance(ref, (int, float)):            ecarts.append(abs(x - ref))if ecarts:    print(f"\necart maximal au protocole temporel, toutes variantes : {max(ecarts):.4f}")    print(f"ecart median                                          : {np.median(ecarts):.4f}")STATE["synthese_ecarts"] = {"max_temporal": float(max(ecarts)) if ecarts else None,                            "median_temporal": float(np.median(ecarts)) if ecarts else None}save_state(STATE)

In [ ]:
# --- 6. Export -----------------------------------------------------------save_state(STATE)print(f"resultats : {STATE_PATH}")print(f"taille    : {STATE_PATH.stat().st_size/1024:.0f} Ko")print("\nA me renvoyer : e4c_results.json")

## Le majeur 11 n'est pas ici, et c'est vouluLe rapport demande cinq graines à 5, 10 et 30 s, contre trois aujourd'hui, pour que lacomparaison entre intervalles soit équilibrée. Cette expérience a besoin du **corpusbrut**, pas du cache 60 s que ce notebook utilise : les intervalles courts n'y sont pas.Elle a donc son propre notebook, **`e5_intervalles_cinq_graines.ipynb`**, qui ne rejouepas les graines 1 à 3 mais vérifie d'abord qu'il les reproduit, puis calcule les seulesgraines 4 et 5. Compter 2 à 3 h, contre 4 à 6 h pour un rejeu intégral.Les deux autres demandes du majeur 11, publier tous les runs individuels et cesser derésumer LightGBM à 10 s par sa moyenne, sont déjà réglées dans le manuscrit : letableau 6 affiche désormais chaque graine, tiret compris là où elle manque.